In [2]:
# %% [markdown]
# # Visualización del progreso de un estudiante
# Este notebook carga un grafo musical y datos sintéticos de un estudiante, y visualiza su progreso paso a paso.

# %%
import networkx as nx
import matplotlib.pyplot as plt
import json
import os
from collections import defaultdict
from matplotlib.colors import to_rgb, to_hex

# %% [markdown]
# ## Rutas de los archivos

# %%
GRAPH_PATH = '../data/graph/nodes.json'
JSONL_PATH = '../data/synthetic/synthetic_data.jsonl'
OUTPUT_DIR = '../data/synthetic/learning_progress/' 

TARGET_STUDENT_ID = 'sint_0501'

# %% [markdown]
# ## Función para cargar el grafo musical

# %%
def load_musical_tree(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

all_nodes_data = load_musical_tree(GRAPH_PATH)

# %% [markdown]
# ## Función para asignar colores suaves según la habilidad

# %%
def get_skill_color(proficiency):
    """
    Colorea según el valor de la habilidad:
    - rojo suave: proficiency < 0
    - amarillo suave: 0.2 <= proficiency <= 0.5
    - verde suave a intenso: 0.5 < proficiency <= 1
    """
    if proficiency < 0:
        return '#F08080'  # rojo suave
    elif 0.2 <= proficiency <= 0.5:
        return '#FFFFE0'  # amarillo suave
    elif 0.5 < proficiency <= 1.0:
        # De verde claro (#C8E6C9) a verde intenso (#8FBC8F)
        factor = (proficiency - 0.5) / 0.5
        r1, g1, b1 = to_rgb('#C8E6C9')  # verde suave
        r2, g2, b2 = to_rgb('#8FBC8F')  # verde más intenso
        r = r1 + factor * (r2 - r1)
        g = g1 + factor * (g2 - g1)
        b = b1 + factor * (b2 - b1)
        return to_hex((r, g, b))
    else:
        return '#D3D3D3'  # gris pendiente

# %% [markdown]
# ## Función para visualizar un árbol de progreso y guardar imagen

# %%
def visualize_learning_path(session_data, all_nodes_data, output_dir=OUTPUT_DIR):
    G = nx.DiGraph()
    
    # Añadir nodos
    for node in all_nodes_data['nodes']:
        G.add_node(node['id'], label=node['name'], level=node['level'])

    node_colors = []
    node_labels = {}

    for node in all_nodes_data['nodes']:
        skill_id = node['id']
        proficiency = session_data['state']['skill_proficiency'].get(skill_id, 0.0)
        color = get_skill_color(proficiency)
        node_colors.append(color)
        node_labels[skill_id] = f"{node['name']}\n({proficiency:.2f})"
    
    # Añadir aristas
    for edge in all_nodes_data['edges']:
        G.add_edge(edge['source'], edge['target'])
    
    # Layout por niveles
    pos = nx.multipartite_layout(G, subset_key='level', align='vertical')
    
    plt.figure(figsize=(15, 12))
    nx.draw(G, pos,
            node_color=node_colors,
            labels=node_labels,
            with_labels=True,
            node_size=2500,
            node_shape='o',
            font_size=8,
            font_weight='bold')
    plt.title(f"Progreso de Sesión {session_data['session_id']} - Paso {session_data.get('step_number', 'N/A')}")
    
    # Crear carpeta específica del estudiante
    student_dir = os.path.join(output_dir, session_data['student_id'])
    os.makedirs(student_dir, exist_ok=True)
    
    # Guardar la imagen en la carpeta del estudiante
    filename = f"{session_data['student_id']}_{session_data['session_id']}_step{session_data.get('step_number', 'N/A')}.png"
    filepath = os.path.join(student_dir, filename)
    plt.savefig(filepath, bbox_inches='tight')
    plt.close()
    
    return filepath  # Retornamos la ruta de la imagen

# %% [markdown]
# ## Función para procesar todas las sesiones de un estudiante y guardar imágenes

# %%
def process_student_sessions(jsonl_path, student_id, all_nodes_data, output_dir=OUTPUT_DIR):
    sessions_by_student = defaultdict(list)
    saved_files = []
    
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            event = json.loads(line)
            if event.get("student_id") == student_id:
                session_id = event.get("session_id")
                sessions_by_student[session_id].append(event)

    if not sessions_by_student:
        print(f"No se encontraron datos para el estudiante {student_id}.")
        return

    # Procesa cada sesión del estudiante
    print(f"Procesando todas las sesiones del estudiante: {student_id}")
    for session_id, events in sessions_by_student.items():
        print(f"\nSesión: {session_id}")
        for i, event in enumerate(events):
            if 'state' in event:
                event['step_number'] = i + 1
                filepath = visualize_learning_path(event, all_nodes_data, output_dir)
                saved_files.append(filepath)
    
    student_dir = os.path.join(output_dir, student_id)
    print(f"\nTodas las imágenes se guardaron en la carpeta del estudiante: {student_dir}")
    for f in saved_files:
        print(f)

# %% [markdown]
# ## Ejecutar la visualización y guardar imágenes

# %%
process_student_sessions(JSONL_PATH, TARGET_STUDENT_ID, all_nodes_data)


Procesando todas las sesiones del estudiante: sint_0501

Sesión: ses_sint_0501_1

Sesión: ses_sint_0501_2

Sesión: ses_sint_0501_3

Sesión: ses_sint_0501_4

Sesión: ses_sint_0501_5

Sesión: ses_sint_0501_6

Sesión: ses_sint_0501_7

Sesión: ses_sint_0501_8

Sesión: ses_sint_0501_9

Sesión: ses_sint_0501_10

Sesión: ses_sint_0501_11

Sesión: ses_sint_0501_12

Sesión: ses_sint_0501_13

Sesión: ses_sint_0501_14

Sesión: ses_sint_0501_15

Sesión: ses_sint_0501_16

Sesión: ses_sint_0501_17

Sesión: ses_sint_0501_18

Todas las imágenes se guardaron en la carpeta del estudiante: ../data/synthetic/learning_progress/sint_0501
../data/synthetic/learning_progress/sint_0501\sint_0501_ses_sint_0501_1_step1.png
../data/synthetic/learning_progress/sint_0501\sint_0501_ses_sint_0501_1_step2.png
../data/synthetic/learning_progress/sint_0501\sint_0501_ses_sint_0501_1_step3.png
../data/synthetic/learning_progress/sint_0501\sint_0501_ses_sint_0501_1_step4.png
../data/synthetic/learning_progress/sint_0501\si